In [ ]:
# import ssl

# # Work around Windows/OpenSSL certificate-store issues that can break aiohttp
# # during the Google GenAI import in this environment.
# try:
#     def _safe_create_default_context(*args, **kwargs):
#         ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
#         ctx.check_hostname = False
#         ctx.verify_mode = ssl.CERT_NONE
#         return ctx

#     ssl.create_default_context = _safe_create_default_context
# except Exception:
#     pass

# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_community.embeddings import HuggingFaceBgeEmbeddings
# from langchain_core.retrievers import BaseRetriever
# from langchain.vectorstores import Chroma
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndPoint
# from langchain.llms import HuggingFacePipeline
# from langchain_core.prompts import PromptTemplate
# from langchain.document_loaders import PyPDFLoader
# from langhain_google_genai.llms import GoogleGenerativeAI
# from langchain_google_genai import GoogleGenerativeAI
from langchain_google_genai.llms import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_core.prompts import PromptTemplate
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_core.retrievers import BaseRetriever
# from langchain.vectorstores import Chroma
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndPoint
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

import torch

import langchain

from huggingface_hub import login
# from google.colab import userdata

import os
from dotenv import load_dotenv


# General Chat Models

In [ ]:
load_dotenv()

# Map your custom .env name to the one LangChain expects
os.environ["GOOGLE_API_KEY"] = os.getenv("gemini")
key = os.getenv("gemini")
# print(key)

## Gemini 2.5 API

In [ ]:
# # Initialize the model
gemini_chat_llm = GoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:


print(gemini_chat_llm.invoke("explain difference between Ai agents and Agentic AIs?"))

In [ ]:
# import langchain
# from langchain.llms import gemini
Llama_HF_key = os.getenv("HF_Token")
if Llama_HF_key:
  login(Llama_HF_key)
# print(Llama_HF_key)

In [ ]:
# from langchain_community.chat_models import ChatHuggingFace

# model_id = "meta-llama/Llama-2-7b-chat-hf"
# model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
model_id = "meta-llama/Llama-3.2-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

## Ollama with Langchain (deepseek-r1:8b)

In [ ]:
from langchain_ollama import ChatOllama


system_prompt = "You are a helpful assistant that can answer questions and return the reponse in markdown format."
message = [
    ("system",system_prompt),
    ("human","explain difference between Ai agents and Agentic AIs?")
]

ollama_models = ChatOllama(
    model= "deepseek-r1:8b",
    validate_model_on_init=True,
    num_ctx= 1050,
    num_predict=1000,
    
)

In [ ]:
for chunk in ollama_models.stream(message):
    print(chunk.text, end="", flush=True)

# RAG

In [ ]:
PAPERS = r"papers"
file_paths = [os.path.join(PAPERS,file)for file in os.listdir(PAPERS)]
# print(file_paths)
docs = []
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=526)

In [ ]:
print(f"Total number of files: {len(file_paths)}")

In [ ]:
for file_path in file_paths[16:]:
    PDF_loder = PyPDFLoader(file_path=file_path)
    PDF_doc = PDF_loder.load()
    # print(PDF_doc)
    docs +=text_spliter.split_documents(PDF_doc)

In [ ]:
from langchain_ollama import OllamaEmbeddings
import os

PERSIST = r'rag practice\vector_db\Chroma'
os.makedirs(PERSIST, exist_ok=True)

# Force the notebook to use the local Ollama server on the standard port.
os.environ.setdefault('OLLAMA_HOST', 'http://127.0.0.1:11434')

embed_model = 'nomic-embed-text:latest'
embed = OllamaEmbeddings(
    model=embed_model,
    # base_url=os.environ['OLLAMA_HOST'],
)

# print(f'Embedding backend: {embed_model} @ {os.environ["OLLAMA_HOST"]}')

In [ ]:
# To create a new vector database or update, uncomment the following lines and run the cell. This will create a new Chroma vector database in the specified PERSIST directory.
chroma_vector_db = Chroma.from_documents(
    embedding=embed,
    documents=docs,
    persist_directory=PERSIST
)

In [ ]:
# To load an existing vector database, uncomment the following lines and run the cell. This will load the Chroma vector database from the specified PERSIST directory.
chroma_vector_db = Chroma(
    embedding_function=embed,
    # documents=docs,
    persist_directory=PERSIST
)

In [ ]:
chroma_vector_db.persist()

In [ ]:
retriver = chroma_vector_db.as_retriever(k=20)

In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

## Ollama RAG

In [ ]:
template = """Use the following pieces of context to answer the question at the end.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"""
prompt = PromptTemplate.from_template(template)

In [ ]:
qa_chain = RetrievalQA.from_llm(
    llm=ollama_models,
    retriever=retriver,
    prompt=prompt,
    # return_source_documents=True,
    )

In [ ]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk.text,end="")

## Gemmini AI RAG

In [ ]:
gemini_rag_template = """You are a personal assistant providing proper explanations for the user. Use the following pieces of context to answer the question at the end.

{context}

Question: {question}
Helpful Answer:"""
gemini_rag_prompt = PromptTemplate.from_template(gemini_rag_template)

gemini_rag = RetrievalQA.from_llm(
    llm=gemini_chat_llm,
    retriever=retriver,
    prompt=gemini_rag_prompt,
    )

In [ ]:
for chunk in gemini_rag.stream({"query":"what is Tg,Tc, FVV, density, Rg in polymer properties, what are the latest advancment in predicting Tg,Tc,Fvv,Rg,density target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="")